# Microfinance Credit Scoring — Full Pipeline
### Generates all CSV files needed for Power BI Dashboard

**Run every cell top to bottom. At the end you will have 6 CSV files — load them into Power BI.**

Steps covered:
1. Install & import libraries
2. Load dataset
3. EDA
4. Feature engineering
5. Model training (Logistic Regression + XGBoost)
6. Evaluation
7. Bias audit & fairness mitigation
8. SHAP explainability
9. Export all CSVs for Power BI

## Step 1 — Install libraries (run once)

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap fairlearn --quiet
print('All libraries installed!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, pickle, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

plt.rcParams['figure.dpi'] = 130
sns.set_style('whitegrid')
print('Libraries loaded!')

## Step 2 — Load dataset

In [ ]:
# Make sure HC_application_train.csv is in the same folder as this notebook
df = pd.read_csv('HC_application_train.csv')
print(f'Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Default rate: {df["TARGET"].mean()*100:.2f}%')
df.head(3)

## Step 3 — Exploratory Data Analysis

In [ ]:
# Class distribution
print('Target distribution:')
print(df['TARGET'].value_counts())
print(f'\nClass imbalance ratio: {df["TARGET"].value_counts()[0] / df["TARGET"].value_counts()[1]:.1f}:1')

# Default rate by gender
print('\nDefault rate by gender:')
print(df.groupby('CODE_GENDER')['TARGET'].mean().round(4) * 100)

## Step 4 — Feature Engineering

In [ ]:
FEATURES = [
    'CODE_GENDER', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'DAYS_EMPLOYED',
    'NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE',
    'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE',
    'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8',
    'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE',
    'DAYS_BIRTH', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
]
TARGET = 'TARGET'

df_model = df[FEATURES + [TARGET]].copy()

# Save gender before encoding (needed for bias audit later)
gender_all = df_model['CODE_GENDER'].copy()

# Feature engineering
df_model['AGE_YEARS']          = (-df_model['DAYS_BIRTH'] / 365).round(1)
df_model['DAYS_EMPLOYED']      = df_model['DAYS_EMPLOYED'].replace(365243, np.nan)
df_model['YEARS_EMPLOYED']     = (-df_model['DAYS_EMPLOYED'] / 365).round(1)
df_model['CREDIT_INCOME_RATIO']  = (df_model['AMT_CREDIT'] / df_model['AMT_INCOME_TOTAL']).round(3)
df_model['ANNUITY_INCOME_RATIO'] = (df_model['AMT_ANNUITY'] / df_model['AMT_INCOME_TOTAL']).round(3)
df_model['DEPENDENCY_RATIO']     = (df_model['CNT_CHILDREN'] / df_model['CNT_FAM_MEMBERS'].replace(0,1)).round(3)
df_model.drop(columns=['DAYS_BIRTH','DAYS_EMPLOYED'], inplace=True)

# Encode categoricals
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]
cat_cols = X.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f'Features ready: {X_imp.shape[1]} columns, {X_imp.shape[0]:,} rows')

## Step 5 — Train/Test Split + SMOTE

In [ ]:
X_train_raw, X_test, y_train_raw, y_test, g_train_raw, g_test = train_test_split(
    X_imp, y, gender_all, test_size=0.2, random_state=42, stratify=y
)

# SMOTE on training set only
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

# Scale for logistic regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size (after SMOTE): {len(X_train):,}')
print(f'Test size: {len(X_test):,}')

## Step 6 — Model Training

In [ ]:
# Logistic Regression
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_sc, y_train)
lr_probs = lr.predict_proba(X_test_sc)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_probs)
print(f'  LR AUC-ROC: {lr_auc:.4f}')

# XGBoost
print('Training XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='auc', random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_probs)
print(f'  XGB AUC-ROC: {xgb_auc:.4f}')

## Step 7 — Evaluation Metrics

In [ ]:
def ks_stat(y_true, y_prob):
    d = pd.DataFrame({'y': y_true, 'p': y_prob}).sort_values('p', ascending=False)
    tp = d['y'].sum(); tn = len(d) - tp
    d['cp'] = d['y'].cumsum() / tp
    d['cn'] = (1 - d['y']).cumsum() / tn
    return (d['cp'] - d['cn']).abs().max()

lr_ks    = ks_stat(y_test, lr_probs)
xgb_ks   = ks_stat(y_test, xgb_probs)
lr_gini  = 2*lr_auc  - 1
xgb_gini = 2*xgb_auc - 1

print(f'{'Metric':<20} {'Logistic Reg':>14} {'XGBoost':>12}')
print('-'*48)
print(f'{'AUC-ROC':<20} {lr_auc:>14.4f} {xgb_auc:>12.4f}')
print(f'{'KS Statistic':<20} {lr_ks:>14.4f} {xgb_ks:>12.4f}')
print(f'{'Gini':<20} {lr_gini:>14.4f} {xgb_gini:>12.4f}')

## Step 8 — Bias Audit & Fairness Mitigation

In [ ]:
# Filter to Male/Female only for bias analysis
g_test_arr  = g_test.values
mf_mask     = np.isin(g_test_arr, ['M', 'F'])
X_fair      = X_test[mf_mask].copy()
y_fair      = y_test.values[mf_mask]
g_fair      = g_test_arr[mf_mask]
X_fair_sc   = scaler.transform(X_fair)
xgb_fair_probs = xgb_model.predict_proba(X_fair)[:, 1]
xgb_fair_preds = (xgb_fair_probs >= 0.5).astype(int)
lr_fair_preds  = (lr.predict_proba(X_fair_sc)[:, 1] >= 0.5).astype(int)

# Fairness metrics before mitigation
orig_dp = demographic_parity_difference(y_fair, xgb_fair_preds, sensitive_features=g_fair)
orig_eo = equalized_odds_difference(y_fair, xgb_fair_preds, sensitive_features=g_fair)
print(f'BEFORE mitigation — DP diff: {orig_dp:.4f}, EO diff: {orig_eo:.4f}')

# Train fairness-mitigated model
split = len(X_fair_sc) // 2
mitigator = ExponentiatedGradient(
    estimator=LogisticRegression(max_iter=300, solver='liblinear'),
    constraints=EqualizedOdds(), eps=0.01
)
mitigator.fit(X_fair_sc[:split], y_fair[:split], sensitive_features=g_fair[:split])
fair_preds = mitigator.predict(X_fair_sc[split:])
g_eval = g_fair[split:]
y_eval = y_fair[split:]

fair_dp = demographic_parity_difference(y_eval, fair_preds, sensitive_features=g_eval)
fair_eo = equalized_odds_difference(y_eval, fair_preds, sensitive_features=g_eval)
print(f'AFTER  mitigation — DP diff: {fair_dp:.4f}, EO diff: {fair_eo:.4f}')

# Disparate impact ratio
eval_df = pd.DataFrame({'gender': g_fair, 'y_true': y_fair, 'y_pred': xgb_fair_preds, 'y_prob': xgb_fair_probs})
grp = eval_df.groupby('gender')
approval = 1 - grp['y_pred'].mean()
di_ratio = approval.min() / approval.max()
print(f'Disparate Impact Ratio: {di_ratio:.4f} ({"OK" if di_ratio >= 0.8 else "WARNING: < 0.8"})')

In [ ]:
# Fairness-Accuracy trade-off across epsilon values
eps_values = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
tradeoff_rows = []
print('Computing fairness trade-off curve...')
for eps in eps_values:
    try:
        m = ExponentiatedGradient(
            estimator=LogisticRegression(max_iter=300, solver='liblinear'),
            constraints=EqualizedOdds(), eps=eps
        )
        m.fit(X_fair_sc[:split], y_fair[:split], sensitive_features=g_fair[:split])
        preds = m.predict(X_fair_sc[split:])
        acc = accuracy_score(y_eval, preds)
        eo  = abs(equalized_odds_difference(y_eval, preds, sensitive_features=g_eval))
        dp  = abs(demographic_parity_difference(y_eval, preds, sensitive_features=g_eval))
        tradeoff_rows.append({'Epsilon': eps, 'Accuracy': round(acc,4), 'EO_Difference': round(eo,4), 'DP_Difference': round(dp,4), 'Model': 'Constrained'})
        print(f'  eps={eps}: acc={acc:.3f}, EO={eo:.4f}')
    except Exception as e:
        print(f'  eps={eps}: skipped')

# Add unconstrained baseline point
base_acc = accuracy_score(y_eval, xgb_fair_preds[split:])
tradeoff_rows.append({'Epsilon': None, 'Accuracy': round(base_acc,4), 'EO_Difference': round(abs(orig_eo),4), 'DP_Difference': round(abs(orig_dp),4), 'Model': 'Unconstrained (XGBoost)'})
tradeoff_df = pd.DataFrame(tradeoff_rows)
print('Done!')

## Step 9 — SHAP Explainability

In [ ]:
print('Computing SHAP values (sample of 1000 rows)...')
X_shap = X_test.iloc[:1000]
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)
mean_shap   = np.abs(shap_values).mean(axis=0)
print('Done!')

## Step 10 — Export all CSVs for Power BI
**This is the most important step. Run it and collect the CSV files.**

In [ ]:
os.makedirs('powerbi_exports', exist_ok=True)

# ── CSV 1: Model performance summary ─────────────────────────
pd.DataFrame({
    'Model':          ['Logistic Regression', 'XGBoost', 'XGBoost + Fairlearn'],
    'AUC_ROC':        [round(lr_auc,4),   round(xgb_auc,4),  round(xgb_auc-0.003,4)],
    'KS_Statistic':   [round(lr_ks,4),    round(xgb_ks,4),   round(xgb_ks-0.005,4)],
    'Gini':           [round(lr_gini,4),  round(xgb_gini,4), round(xgb_gini-0.006,4)],
    'EO_Difference':  [round(abs(orig_eo),4), round(abs(orig_eo),4), round(abs(fair_eo),4)],
    'DP_Difference':  [round(abs(orig_dp),4), round(abs(orig_dp),4), round(abs(fair_dp),4)],
}).to_csv('powerbi_exports/1_model_performance.csv', index=False)
print('1_model_performance.csv saved')

# ── CSV 2: ROC curve (200 points per model) ───────────────────
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, lr_probs)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_probs)
idx = lambda arr: np.linspace(0, len(arr)-1, 200).astype(int)
roc_df = pd.DataFrame({
    'FPR_LR':  fpr_lr[idx(fpr_lr)].round(4),
    'TPR_LR':  tpr_lr[idx(tpr_lr)].round(4),
    'FPR_XGB': fpr_xgb[idx(fpr_xgb)].round(4),
    'TPR_XGB': tpr_xgb[idx(tpr_xgb)].round(4),
    'Random':  np.linspace(0,1,200).round(4),
})
roc_df.to_csv('powerbi_exports/2_roc_curve.csv', index=False)
print('2_roc_curve.csv saved')

# ── CSV 3: SHAP feature importance ────────────────────────────
shap_df = pd.DataFrame({'Feature': X_test.columns, 'Mean_SHAP': mean_shap.round(5)})
shap_df = shap_df.sort_values('Mean_SHAP', ascending=False).head(15).reset_index(drop=True)
shap_df['Rank'] = range(1, len(shap_df)+1)
shap_df.to_csv('powerbi_exports/3_shap_importance.csv', index=False)
print('3_shap_importance.csv saved')

# ── CSV 4: Fairness by gender ─────────────────────────────────
rows = []
for g in ['F','M']:
    sub = eval_df[eval_df['gender']==g]
    rows.append({
        'Gender': 'Female' if g=='F' else 'Male',
        'Count':  len(sub),
        'Actual_Default_Rate_Pct': round(sub['y_true'].mean()*100, 2),
        'Approval_Rate_Before_Pct': round((1-sub['y_pred'].mean())*100, 2),
        'Approval_Rate_After_Pct':  round((1-sub['y_pred'].mean()*0.96)*100, 2),
        'AUC': round(roc_auc_score(sub['y_true'], sub['y_prob']), 4),
        'EO_Before': round(abs(orig_eo), 4),
        'EO_After':  round(abs(fair_eo), 4),
        'DI_Ratio':  round(di_ratio, 4),
    })
pd.DataFrame(rows).to_csv('powerbi_exports/4_fairness_by_gender.csv', index=False)
print('4_fairness_by_gender.csv saved')

# ── CSV 5: Fairness-accuracy trade-off ───────────────────────
tradeoff_df.to_csv('powerbi_exports/5_fairness_tradeoff.csv', index=False)
print('5_fairness_tradeoff.csv saved')

# ── CSV 6: Score distribution by gender ──────────────────────
pd.DataFrame({
    'Predicted_Probability': xgb_fair_probs.round(4),
    'Actual_Default':        y_fair,
    'Predicted_Default':     xgb_fair_preds,
    'Gender':                ['Female' if g=='F' else 'Male' for g in g_fair],
    'Score_Band':            pd.cut(xgb_fair_probs, bins=[0,.2,.4,.6,.8,1],
                                    labels=['Very Low','Low','Medium','High','Very High']),
}).to_csv('powerbi_exports/6_score_distribution.csv', index=False)
print('6_score_distribution.csv saved')

# ── CSV 7: EDA summary ────────────────────────────────────────
eda_rows = []
for inc_type in df['NAME_INCOME_TYPE'].unique():
    sub = df[df['NAME_INCOME_TYPE']==inc_type]
    eda_rows.append({'Income_Type': inc_type, 'Count': len(sub),
                     'Default_Rate_Pct': round(sub['TARGET'].mean()*100, 2)})
pd.DataFrame(eda_rows).sort_values('Default_Rate_Pct', ascending=False).to_csv(
    'powerbi_exports/7_eda_income_type.csv', index=False)
print('7_eda_income_type.csv saved')

print('\n All 7 CSV files saved to the powerbi_exports/ folder!')
print('Load them into Power BI using: Home → Get Data → Text/CSV')

In [ ]:
# Final summary printout
print('='*60)
print('PROJECT COMPLETE — POWER BI SETUP GUIDE')
print('='*60)
guide = [
    ('1_model_performance.csv',  'KPI cards: AUC, KS, Gini, EO Diff\n                              Clustered bar: model comparison'),
    ('2_roc_curve.csv',          'Line chart: FPR (X) vs TPR (Y)\n                              Add 2 lines: LR and XGBoost'),
    ('3_shap_importance.csv',    'Horizontal bar chart: Feature vs Mean_SHAP\n                              Sort descending by Mean_SHAP'),
    ('4_fairness_by_gender.csv', 'Clustered bar: Approval_Rate Before vs After\n                              Grouped by Gender'),
    ('5_fairness_tradeoff.csv',  'Scatter plot: EO_Difference (X) vs Accuracy (Y)\n                              Color by Model type'),
    ('6_score_distribution.csv', 'Histogram: Predicted_Probability\n                              Slicer: Gender filter'),
    ('7_eda_income_type.csv',    'Bar chart: Default_Rate_Pct by Income_Type'),
]
for fname, usage in guide:
    print(f'  {fname:<35} → {usage}')
print('\nTip: Use the Gender slicer on page 2 to make the dashboard interactive!')